# SENet（CIFAR100を用いた画像認識）

---
## 目的
畳み込みニューラルネットワークにチャネル方向のアテンション機構を導入したSqueeze-and-Excitation Network（SENet）を用いて，CIFAR100データセットの100クラス物体認識を行う．ResNetのResidual BlockにSEブロックを組み込むことで，特徴マップのチャネルごとの重要度を適応的に調整する仕組みについて理解する．

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
from time import time

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchsummary

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
CIFAR100データセットを読み込みます．CIFAR10と同様32×32ピクセルのカラー画像データセットですが，クラス数が100とより細かく分類されています．学習用データには`RandomCrop`と`RandomHorizontalFlip`によるData Augmentationを適用します．

In [ ]:
transform_train = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                       transforms.RandomHorizontalFlip(),
                                       transforms.ToTensor()])
transform_test = transforms.Compose([transforms.ToTensor()])

train_data = torchvision.datasets.CIFAR100(root="./", train=True, transform=transform_train, download=True)
test_data = torchvision.datasets.CIFAR100(root="./", train=False, transform=transform_test, download=True)

print(len(train_data), len(test_data))

## Squeeze-and-Excitationブロック
通常の畳み込み層は，全てのチャンネルを等しく扱って特徴マップを出力します．しかし，認識に重要なチャンネルとそうでないチャンネルは入力画像によって異なるはずです．SENetは，このチャンネルごとの重要度を学習によって求め，特徴マップをチャンネル単位で重み付けする**Squeeze-and-Excitation（SE）ブロック**を提案しました．

SEブロックは，次の2段階の処理から構成されます．

- **Squeeze**：Global Average Poolingにより，各チャンネルの空間方向の情報を1つのスカラー値に集約する．
- **Excitation**：Squeezeで得たチャンネルごとの値を，全結合層2つからなる小さなボトルネック構造（削減率$r$で一度チャンネル数を絞り，ReLUを通したのち元のチャンネル数に戻す）とSigmoid関数に通し，各チャンネルの重要度を$0〜1$の重みとして出力する．

得られた重みを元の特徴マップにチャンネルごとに掛け合わせることで，重要なチャンネルを強調し，不要なチャンネルを抑制します．全結合層のみで構成されるため計算コストは小さく，既存のネットワークに追加しやすいという特徴があります．削減率$r$（`reduction`）はSEブロックのパラメータ数と表現力のトレードオフを決めるハイパーパラメータで，元論文では16が標準的に使われています．

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, channel, reduction=16):
        super().__init__()
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        w = self.avgpool(x).view(b, c)    # Squeeze: チャンネルごとに空間方向を平均化
        w = self.fc(w).view(b, c, 1, 1)   # Excitation: チャンネルごとの重みを算出
        return x * w                      # チャンネルごとに特徴マップを再重み付け

## SE-ResNet：Residual BlockへのSEブロックの組み込み
SEブロックは単体では使わず，既存のネットワークの各ブロックに追加する形で使用します．ここでは，`resnet.ipynb`で実装したBasicBlockとBottleneckのResidual Blockを再利用し，畳み込み層による変換の出力に対してSEブロックでチャンネルを再重み付けしてから，ショートカットとの加算を行う「SE-ResNet」を構築します．

`resnet.ipynb`で説明した通り，ResNetには224×224ピクセルのImageNet画像を対象とした「ImageNet版」と，32×32ピクセルのCIFAR10/100を対象とした「CIFAR版」があります．本ノートブックでは，`resnet.ipynb`のCIFAR版ResNetをベースにSEブロックを追加したCIFAR版のSE-ResNetを実装します（詳細は本ノートブック末尾の「ImageNet版SENetとCIFAR版SENetの違い」を参照）．

In [ ]:
class SEBasicBlock(nn.Module):
    expansion = 1
    def __init__(self, inplanes, planes, stride=1, downsample=None, reduction=16):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Conv2d(inplanes, planes, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(planes),
            nn.ReLU(inplace=True),
            nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(planes),
        )
        self.se = SEBlock(planes * self.expansion, reduction)
        self.downsample = downsample
        self.relu = nn.ReLU(inplace=True)
        self.stride = stride

    def forward(self, x):
        residual = x
        out = self.convs(x)
        out = self.se(out)  # SEブロックによるチャンネル方向の再重み付け
        if self.downsample is not None:
            residual = self.downsample(x)
        out += residual
        out = self.relu(out)
        return out


class SEBottleneck(nn.Module):
    expansion = 4
    def __init__(self, inplanes, planes, stride=1, downsample=None, reduction=16):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Conv2d(inplanes, planes, kernel_size=1, bias=False),
            nn.BatchNorm2d(planes),
            nn.ReLU(inplace=True),
            nn.Conv2d(planes, planes, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(planes),
            nn.ReLU(inplace=True),
            nn.Conv2d(planes, self.expansion * planes, kernel_size=1, bias=False),
            nn.BatchNorm2d(self.expansion * planes),
        )
        self.se = SEBlock(self.expansion * planes, reduction)
        self.downsample = downsample
        self.relu = nn.ReLU(inplace=True)
        self.stride = stride

    def forward(self, x):
        residual = x
        out = self.convs(x)
        out = self.se(out)  # SEブロックによるチャンネル方向の再重み付け
        if self.downsample is not None:
            residual = self.downsample(x)
        out += residual
        out = self.relu(out)
        return out

### SE-ResNetの定義
`resnet.ipynb`と同様に，`depth`（畳み込みの層数）から1ブロックあたりのResidual Block数を算出してネットワークを構築します．BasicBlockを用いる場合は`depth = 6n + 2`，Bottleneckを用いる場合は`depth = 9n + 2`という関係になります．削減率`reduction`はコンストラクタの引数として指定でき，各Residual Block内のSEブロックに渡されます．

In [ ]:
class SEResNetBasicBlock(nn.Module):
    def __init__(self, depth, n_class=100, reduction=16):
        super().__init__()
        assert (depth - 2) % 6 == 0, 'When use SEBasicBlock, depth should be 6n+2 (e.g. 20, 32, 44).'
        n_blocks = (depth - 2) // 6
        self.reduction = reduction

        self.inplanes = 16
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(16, n_blocks)
        self.layer2 = self._make_layer(32, n_blocks, stride=2)
        self.layer3 = self._make_layer(64, n_blocks, stride=2)

        self.avgpool = nn.AvgPool2d(8)
        self.fc = nn.Linear(64 * SEBasicBlock.expansion, n_class)

    def _make_layer(self, planes, n_blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * SEBasicBlock.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * SEBasicBlock.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * SEBasicBlock.expansion),
            )

        layers = [SEBasicBlock(self.inplanes, planes, stride, downsample, self.reduction)]
        self.inplanes = planes * SEBasicBlock.expansion
        for _ in range(n_blocks - 1):
            layers.append(SEBasicBlock(self.inplanes, planes, reduction=self.reduction))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


class SEResNetBottleneck(nn.Module):
    def __init__(self, depth, n_class=100, reduction=16):
        super().__init__()
        assert (depth - 2) % 9 == 0, 'When use SEBottleneck, depth should be 9n+2 (e.g. 47, 56, 110).'
        n_blocks = (depth - 2) // 9
        self.reduction = reduction

        self.inplanes = 16
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(16, n_blocks)
        self.layer2 = self._make_layer(32, n_blocks, stride=2)
        self.layer3 = self._make_layer(64, n_blocks, stride=2)

        self.avgpool = nn.AvgPool2d(8)
        self.fc = nn.Linear(64 * SEBottleneck.expansion, n_class)

    def _make_layer(self, planes, n_blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * SEBottleneck.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * SEBottleneck.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * SEBottleneck.expansion),
            )

        layers = [SEBottleneck(self.inplanes, planes, stride, downsample, self.reduction)]
        self.inplanes = planes * SEBottleneck.expansion
        for _ in range(n_blocks - 1):
            layers.append(SEBottleneck(self.inplanes, planes, reduction=self.reduction))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

## ネットワークの作成
定義したネットワークを作成します．`n_layers`でSE-ResNetの層数を，`reduction`でSEブロックの削減率$r$を指定します．`.to(device)`によりモデルのパラメータを`device`上に配置し，最適化手法にはモーメンタムSGD（学習率0.01，モーメンタム0.9，weight decay 5e-4）を用います．最後に`torchsummary.summary()`でネットワークの詳細情報を表示します．

In [ ]:
# SE-ResNetの層数と削減率を指定 (e.g. 20, 32, 44, 56, 110)
n_layers = 20
reduction = 16

model = SEResNetBasicBlock(depth=n_layers, n_class=100, reduction=reduction)    # SEBasicBlock構造を用いる場合
# model = SEResNetBottleneck(depth=n_layers, n_class=100, reduction=reduction)  # SEBottleneck構造を用いる場合
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

torchsummary.summary(model, (3, 32, 32), device=device.type)

## 学習
読み込んだCIFAR100データセットと作成したネットワークを用いて学習を行います．ミニバッチサイズを128，学習エポック数を20とします．

In [ ]:
batch_size = 128
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)

criterion = nn.CrossEntropyLoss()
criterion = criterion.to(device)

model.train()

train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0
    count = 0

    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        loss = criterion(y, label)
        model.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

    print("epoch: {}, mean loss: {}, mean accuracy: {}, elapsed time: {}".format(
        epoch, sum_loss / len(train_loader), count.item() / len(train_data), time() - train_start))

## テスト
学習したネットワークモデルを用いて評価を行います．

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=100, shuffle=False)

model.eval()

count = 0
with torch.no_grad():
    for image, label in test_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

print("test accuracy: {}".format(count.item() / len(test_data)))

## ImageNet版SENetとCIFAR版SENetの違い
このノートブックで実装したSE-ResNetは，`resnet.ipynb`のCIFAR版ResNetをベースにSEブロックを追加したものです．広く知られているImageNet版SENetとは，主に次の点が異なります．

| | ImageNet版 | CIFAR版（本ノートブック） |
|---|---|---|
| ベースとなるResNet構造 | ImageNet版ResNet（1層目が7×7畳み込み，特徴マップサイズごとに4ブロック） | CIFAR版ResNet（1層目が3×3畳み込み，特徴マップサイズごとに3ブロック） |
| SEブロックの挿入位置 | 各Residual Blockの畳み込み後・ショートカット加算前 | 同左（変更なし） |
| 削減率$r$ | 16（論文の既定値） | 16（既定値，`reduction`引数で変更可能） |
| 出力層のクラス数 | 1000 | 100（CIFAR100） |

## 課題

1. 削減率`reduction`（$r$）を4，8，16，32などに変更し，パラメータ数と認識精度がどのように変化するか確認しましょう．

2. `SEBasicBlock`の代わりに`BasicBlock`（`resnet.ipynb`参照）を用いた通常のResNetを学習し，SEブロックの有無で認識精度や学習曲線がどのように異なるか比較しましょう．

3. SEブロックを適用する位置を変更（例えば，ショートカットとの加算後に適用する）し，本ノートブックの構成との違いを比較しましょう．